# Process Bigraph paper — figures

_Investigation `paper-figures` — coder reproduction notebook._

One study per figure of the Process Bigraph paper. Each study describes the figure and checks that its bigraph-loom image is generated (shown under Visualizations as an SVG). Fig 2 is conceptual (no composite). Figs 7.1–7.3 and Fig 8 are the runnable spatio-flux composites; some paper figures (e.g. Figs 5–6) live elsewhere and are not built here.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/spatio-flux/spatio-flux').is_dir():
    REPO = Path('/home/runner/work/spatio-flux/spatio-flux')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from spatio_flux.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Fig 1 — draft processes + multiscale composite (`fig-01`)

**Purpose.** Four unwired draft-process cards (Gene Expression, Metabolism, Morphogen Gradient, Multicellular Interactions) and the multiscale draft composite (tissue > cells > cell > molecules), rendered in bigraph-loom.

**Claim.** image generated


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `1a-draft-processes` | `spatio_flux.composites.fig01a-draft-processes` | 0 | — |
| `1b-multiscale` | `spatio_flux.composites.fig01b-multiscale-composite` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `spatio_flux.composites.fig01a-draft-processes`** — `spec_spatio_flux_composites_fig01a_draft_processes` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig01a_draft_processes = load_spec(REPO / 'spatio_flux/composites/fig01a-draft-processes.composite.json')
describe_spec(spec_spatio_flux_composites_fig01a_draft_processes)

In [ ]:
# === Edit parameters for composite 'fig01a-draft-processes' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'gene_expression'  (local:GeneExpression)
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['summary'] = 'Gene expression — ordinary differential equations'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['status'] = 'draft - no update dynamics yet'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['summary'] = 'Gene expression — ordinary differential equations'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['method'] = 'ODE'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['description'] = 'Transcription + translation as coupled ODEs: DNA templates mRNA; mRNA templates protein; each species turns over.'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['math'] = ['\\frac{d\\,\\text{mrna}}{dt} = \\alpha\\,\\text{dna} - \\gamma_m\\,\\text{mrna}', '\\frac{d\\,\\text{protein}}{dt} = \\beta\\,\\text{mrna} - \\gamma_p\\,\\text{protein}']
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['symbols']['dna'] = 'DNA template (in)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['symbols']['energy'] = 'ATP/GTP — powers α, β (in)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['symbols']['mrna'] = 'mRNA (out)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['symbols']['protein'] = 'protein (out)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['symbols']['α'] = 'transcription rate'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['symbols']['β'] = 'translation rate'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['symbols']['γ_m, γ_p'] = 'mRNA / protein turnover rates'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['ports']['dna'] = 'DNA template'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['ports']['energy'] = 'ATP / GTP'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['ports']['mrna'] = 'mRNA'
spec_spatio_flux_composites_fig01a_draft_processes['state']['gene_expression']['config']['contract']['ports']['protein'] = 'protein'

# process 'metabolism'  (local:Metabolism)
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['summary'] = 'Metabolism — Flux Balance Analysis (FBA)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['status'] = 'draft - no update dynamics yet'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['summary'] = 'Metabolism — Flux Balance Analysis (FBA)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['method'] = 'Flux Balance Analysis'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['description'] = 'Constraint-based steady-state flux optimization: maximize a biomass / objective flux subject to mass balance and flux bounds. Enzyme levels and energy set the bounds.'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['math'] = ['\\text{maximize}\\quad Z = c^{\\mathsf{T}} v \\quad\\text{s.t.}\\quad S\\,v = 0', 'v_{\\min} \\le v \\le v_{\\max}(\\text{enzymes},\\,\\text{energy})', '\\Delta\\,\\text{metabolites} = S_{\\text{ex}}\\,v\\,\\Delta t']
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['symbols']['enzymes'] = 'enzyme levels — set flux bounds (in)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['symbols']['energy'] = 'available energy — sets bounds (in)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['symbols']['metabolites'] = 'produced metabolites (out)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['symbols']['Z'] = 'biomass objective'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['symbols']['v'] = 'reaction fluxes'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['symbols']['S'] = 'stoichiometric matrix'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['ports']['enzymes'] = 'enzyme (protein) levels'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['ports']['energy'] = 'available energy'
spec_spatio_flux_composites_fig01a_draft_processes['state']['metabolism']['config']['contract']['ports']['metabolites'] = 'produced metabolites'

# process 'morphogen_gradient'  (local:Diffusion)
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['summary'] = 'Morphogen gradient — reaction–diffusion PDE'
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['status'] = 'draft - no update dynamics yet'
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['summary'] = 'Morphogen gradient — reaction–diffusion PDE'
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['method'] = 'Reaction–Diffusion'
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['description'] = 'A diffusing morphogen field with a local source and first-order decay sets up a spatial gradient across the tissue.'
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['math'] = ['\\frac{\\partial\\,\\text{field}}{\\partial t} = D\\,\\nabla^2\\,\\text{field} + S(x) - \\lambda\\,\\text{field}']
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['symbols']['field'] = 'morphogen concentration (in + out)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['symbols']['D'] = 'diffusion coefficient'
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['symbols']['S(x)'] = 'local source'
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['symbols']['λ'] = 'decay rate'
spec_spatio_flux_composites_fig01a_draft_processes['state']['morphogen_gradient']['config']['contract']['ports']['field'] = 'morphogen concentration field'

# process 'multicellular_interactions'  (local:ABM)
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['summary'] = 'Multicellular interactions — Agent-Based Model'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['status'] = 'draft - no update dynamics yet'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['summary'] = 'Multicellular interactions — Agent-Based Model'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['method'] = 'Agent-Based Model'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['description'] = 'Off-lattice agents move under interaction forces, chemotaxis up the morphogen gradient, and stochastic noise; contact-range interactions can remove cells.'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['math'] = ['\\vec{x}_i(t{+}\\Delta t) = \\vec{x}_i + \\left[\\mu\\, f_{\\text{int}} + \\chi\\, \\nabla\\text{field}\\right]\\Delta t + \\sqrt{2 D_m\\,\\Delta t}\\;\\xi_i', 'P_{\\text{kill}} = \\left(1 - e^{-k_{\\text{kill}}\\,\\Delta t}\\right)\\mathbf{1}_{\\lVert \\vec{x}_i - \\vec{x}_j \\rVert < d}']
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['population'] = 'cells {x⃗ᵢ} (in + out)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['field'] = 'morphogen — drives ∇field (in)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['x⃗ᵢ'] = 'position of cell i'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['μ'] = 'mobility'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['f_int'] = 'interaction force'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['χ'] = 'chemotactic coefficient'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['D_m'] = 'motility diffusion'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['ξᵢ'] = 'unit Gaussian noise'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['k_kill'] = 'kill rate'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['symbols']['d'] = 'interaction radius'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['ports']['population'] = 'cell population {x⃗ᵢ}'
spec_spatio_flux_composites_fig01a_draft_processes['state']['multicellular_interactions']['config']['contract']['ports']['field'] = 'local morphogen field'

# process 'neural_dynamics'  (local:NeuralDynamics)
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['summary'] = 'Learned dynamics — neural-network surrogate'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['status'] = 'draft - no update dynamics yet'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['summary'] = 'Learned dynamics — neural-network surrogate'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['method'] = 'Neural Network'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['description'] = "A machine-learning formalism: a neural network f_θ, trained on trajectory data, predicts the state's time-derivative (a neural ODE) — a fast, differentiable stand-in for an unknown or expensive mechanism."
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['math'] = ['\\frac{d\\,\\text{state}}{dt} = f_\\theta(\\text{state}),\\quad f_\\theta = \\mathrm{NN}(\\theta)', '\\theta^\\ast = \\arg\\min_\\theta \\sum_k \\lVert \\widehat{\\text{state}}_\\theta(t_k) - \\text{state}_k \\rVert^2']
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['symbols']['state'] = 'system state (in + out)'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['symbols']['f_θ'] = 'neural network — learned vector field'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['symbols']['θ'] = 'network weights'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['symbols']['NN'] = 'neural network'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['symbols']['ŝtate_θ(t_k)'] = 'model prediction'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['symbols']['state_k'] = 'observed state at t_k'
spec_spatio_flux_composites_fig01a_draft_processes['state']['neural_dynamics']['config']['contract']['ports']['state'] = 'system state'

**Composite `spatio_flux.composites.fig01b-multiscale-composite`** — `spec_spatio_flux_composites_fig01b_multiscale_composite` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig01b_multiscale_composite = load_spec(REPO / 'spatio_flux/composites/fig01b-multiscale-composite.composite.json')
describe_spec(spec_spatio_flux_composites_fig01b_multiscale_composite)

In [ ]:
# === Edit parameters for composite 'fig01b-multiscale-composite' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-01 ===
STUDY = 'fig-01'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fig01a-draft-processes.svg**


In [ ]:
# fig01a-draft-processes.svg
show_viz(_render_one('image:visualizations/fig01a-draft-processes.svg', {}, RUNS_DB, STUDY_YAML))

**fig01b-multiscale-composite.svg**


In [ ]:
# fig01b-multiscale-composite.svg
show_viz(_render_one('image:visualizations/fig01b-multiscale-composite.svg', {}, RUNS_DB, STUDY_YAML))

**1c — study workflow**


In [ ]:
# 1c — study workflow
show_viz(_render_one('image:visualizations/fig01c-study-workflow.svg', {}, RUNS_DB, STUDY_YAML))

**Figure 1 (composite)**


In [ ]:
# Figure 1 (composite)
show_viz(_render_one('image:visualizations/figure_1.svg', {}, RUNS_DB, STUDY_YAML))

## Study: Fig 2 — Milner bigraph (a) + process bigraph (b) (`fig-02`)

**Purpose.** Contrasts a Milner bigraph (a) with a process bigraph (b). The process-bigraph panel is illustrated by the process graph (metabolism + gene expression over shared stores), rendered in bigraph-loom.

**Claim.** image generated


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `2b-process-bigraph` | `spatio_flux.composites.fig02b-processes` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `spatio_flux.composites.fig02b-processes`** — `spec_spatio_flux_composites_fig02b_processes` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig02b_processes = load_spec(REPO / 'spatio_flux/composites/fig02b-processes.composite.json')
describe_spec(spec_spatio_flux_composites_fig02b_processes)

In [ ]:
# === Edit parameters for composite 'fig02b-processes' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'p1'  (local:BigraphLink)
spec_spatio_flux_composites_fig02b_processes['state']['p1']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig02b_processes['state']['p1']['config']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_spatio_flux_composites_fig02b_processes['state']['p1']['config']['contract']['status'] = 'draft - no update'
spec_spatio_flux_composites_fig02b_processes['state']['p1']['config']['contract']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_spatio_flux_composites_fig02b_processes['state']['p1']['config']['contract']['description'] = 'A process in the process bigraph: it connects nodes of the place graph through its typed ports, replacing a Milner hyperedge in the link graph.'
spec_spatio_flux_composites_fig02b_processes['state']['p1']['config']['contract']['ports']['in'] = 'a node this process reads'
spec_spatio_flux_composites_fig02b_processes['state']['p1']['config']['contract']['ports']['out'] = 'a node this process writes'
spec_spatio_flux_composites_fig02b_processes['state']['p1']['config']['contract']['ports']['out_b'] = 'a second node this process writes'

# process 'p2'  (local:BigraphLink)
spec_spatio_flux_composites_fig02b_processes['state']['p2']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig02b_processes['state']['p2']['config']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_spatio_flux_composites_fig02b_processes['state']['p2']['config']['contract']['status'] = 'draft - no update'
spec_spatio_flux_composites_fig02b_processes['state']['p2']['config']['contract']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_spatio_flux_composites_fig02b_processes['state']['p2']['config']['contract']['description'] = 'A process in the process bigraph: it connects nodes of the place graph through its typed ports, replacing a Milner hyperedge in the link graph.'
spec_spatio_flux_composites_fig02b_processes['state']['p2']['config']['contract']['ports']['in'] = 'a node this process reads'
spec_spatio_flux_composites_fig02b_processes['state']['p2']['config']['contract']['ports']['out'] = 'a node this process writes'
spec_spatio_flux_composites_fig02b_processes['state']['p2']['config']['contract']['ports']['out_b'] = 'a second node this process writes'

# process 'p3'  (local:BigraphLink)
spec_spatio_flux_composites_fig02b_processes['state']['p3']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig02b_processes['state']['p3']['config']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_spatio_flux_composites_fig02b_processes['state']['p3']['config']['contract']['status'] = 'draft - no update'
spec_spatio_flux_composites_fig02b_processes['state']['p3']['config']['contract']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_spatio_flux_composites_fig02b_processes['state']['p3']['config']['contract']['description'] = 'A process in the process bigraph: it connects nodes of the place graph through its typed ports, replacing a Milner hyperedge in the link graph.'
spec_spatio_flux_composites_fig02b_processes['state']['p3']['config']['contract']['ports']['in'] = 'a node this process reads'
spec_spatio_flux_composites_fig02b_processes['state']['p3']['config']['contract']['ports']['out'] = 'a node this process writes'
spec_spatio_flux_composites_fig02b_processes['state']['p3']['config']['contract']['ports']['out_b'] = 'a second node this process writes'

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-02 ===
STUDY = 'fig-02'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**2a — Milner hyperedges**


In [ ]:
# 2a — Milner hyperedges
show_viz(_render_one('image:visualizations/fig02a-hyperedges.svg', {}, RUNS_DB, STUDY_YAML))

**2b — process bigraph**


In [ ]:
# 2b — process bigraph
show_viz(_render_one('image:visualizations/fig02b-processes.svg', {}, RUNS_DB, STUDY_YAML))

**Figure 2a (hypergraph)**


In [ ]:
# Figure 2a (hypergraph)
show_viz(_render_one('image:visualizations/figure_2a.svg', {}, RUNS_DB, STUDY_YAML))

**Figure 2b (process bigraph)**


In [ ]:
# Figure 2b (process bigraph)
show_viz(_render_one('image:visualizations/figure_2b.svg', {}, RUNS_DB, STUDY_YAML))

**Figure 2 (hypergraph + process bigraph)**


In [ ]:
# Figure 2 (hypergraph + process bigraph)
show_viz(_render_one('image:visualizations/figure_2.svg', {}, RUNS_DB, STUDY_YAML))

## Study: Fig 3 — process graph + composite process (`fig-03`)

**Purpose.** A process graph (metabolism + gene expression over metab/enzymes/DNA) and the `cell` composite process (cyto/mem/nuc/DNA + grow/express/transport), rendered in bigraph-loom.

**Claim.** image generated


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `3a-store` | `spatio_flux.composites.fig03a-store` | 0 | — |
| `3b-place-graph` | `spatio_flux.composites.fig03b-place-graph` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `spatio_flux.composites.fig03a-store`** — `spec_spatio_flux_composites_fig03a_store` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig03a_store = load_spec(REPO / 'spatio_flux/composites/fig03a-store.composite.json')
describe_spec(spec_spatio_flux_composites_fig03a_store)

In [ ]:
# === Edit parameters for composite 'fig03a-store' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

**Composite `spatio_flux.composites.fig03b-place-graph`** — `spec_spatio_flux_composites_fig03b_place_graph` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig03b_place_graph = load_spec(REPO / 'spatio_flux/composites/fig03b-place-graph.composite.json')
describe_spec(spec_spatio_flux_composites_fig03b_place_graph)

In [ ]:
# === Edit parameters for composite 'fig03b-place-graph' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-03 ===
STUDY = 'fig-03'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**3a — store (full detail)**


In [ ]:
# 3a — store (full detail)
show_viz(_render_one('image:visualizations/fig03a-store.svg', {}, RUNS_DB, STUDY_YAML))

**3b — place graph**


In [ ]:
# 3b — place graph
show_viz(_render_one('image:visualizations/fig03b-place-graph.svg', {}, RUNS_DB, STUDY_YAML))

**Figure 3 (composite)**


In [ ]:
# Figure 3 (composite)
show_viz(_render_one('image:visualizations/figure_3.svg', {}, RUNS_DB, STUDY_YAML))

## Study: Fig 4 — process diagram (`fig-04`)

**Purpose.** A single process shown as a rectangle with typed ports on its boundary — inputs in_1 (species), in_2 (params) on the left; outputs out_1 (ss_species), out_2 (rates) on the right; config type steady_state; the update function maps (in_1, in_2) → (out_1, out_2).

**Claim.** image generated


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `4-process` | `spatio_flux.composites.fig04-process` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `spatio_flux.composites.fig04-process`** — `spec_spatio_flux_composites_fig04_process` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig04_process = load_spec(REPO / 'spatio_flux/composites/fig04-process.composite.json')
describe_spec(spec_spatio_flux_composites_fig04_process)

In [ ]:
# === Edit parameters for composite 'fig04-process' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'process'  (local:ProcessSchematic)
spec_spatio_flux_composites_fig04_process['state']['process']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig04_process['state']['process']['config']['summary'] = 'A typed function signature whose update method emits a delta Δ'
spec_spatio_flux_composites_fig04_process['state']['process']['config']['contract']['status'] = 'draft - no update dynamics yet'
spec_spatio_flux_composites_fig04_process['state']['process']['config']['contract']['summary'] = 'A typed function signature whose update method emits a delta Δ'
spec_spatio_flux_composites_fig04_process['state']['process']['config']['contract']['description'] = 'Typed input ports (what it reads) and typed output ports (what it updates); the update method maps config + inputs to a delta Δ — the tree of changes to apply, branched by output port.'
spec_spatio_flux_composites_fig04_process['state']['process']['config']['contract']['math'] = ['p_{\\text{proc}}\\big[\\text{interval}{:}\\text{integer}\\big]\\ :\\ \\text{in}_1^{\\text{species}},\\ \\text{in}_2^{\\text{params}}\\ \\longrightarrow\\ \\Delta{=}\\{\\text{out}_1^{\\text{ss\\_species}},\\ \\text{out}_2^{\\text{rates}}\\}']
spec_spatio_flux_composites_fig04_process['state']['process']['config']['contract']['ports']['in_1'] = 'species'
spec_spatio_flux_composites_fig04_process['state']['process']['config']['contract']['ports']['in_2'] = 'params'
spec_spatio_flux_composites_fig04_process['state']['process']['config']['contract']['ports']['out_1'] = 'ss_species'
spec_spatio_flux_composites_fig04_process['state']['process']['config']['contract']['ports']['out_2'] = 'rates'

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-04 ===
STUDY = 'fig-04'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**4 — process schematic**


In [ ]:
# 4 — process schematic
show_viz(_render_one('image:visualizations/fig04-process.svg', {}, RUNS_DB, STUDY_YAML))

**Figure 4 (composite)**


In [ ]:
# Figure 4 (composite)
show_viz(_render_one('image:visualizations/figure_4.svg', {}, RUNS_DB, STUDY_YAML))

## Study: Fig 5 — process graph & composite process (`fig-05`)

**Purpose.** Two readings of composition. 5a — a process graph: processes (metabolism, gene_expression) connected to shared stores (metab, enzymes, DNA) through typed ports, with arrow directions showing the input/output wiring. 5b — a composite process: a `cell` process exposing external ports (nutrients, signals in; shape out) that bridge to an internal process bigraph (express / grow / transport over cyto / mem / nuc stores), rendered as a drillable sub-composite.

**Claim.** image generated


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `5a-process-graph` | `spatio_flux.composites.fig05a-process-graph` | 0 | — |
| `5b-composite-process` | `spatio_flux.composites.fig05b-composite-process` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `spatio_flux.composites.fig05a-process-graph`** — `spec_spatio_flux_composites_fig05a_process_graph` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig05a_process_graph = load_spec(REPO / 'spatio_flux/composites/fig05a-process-graph.composite.json')
describe_spec(spec_spatio_flux_composites_fig05a_process_graph)

In [ ]:
# === Edit parameters for composite 'fig05a-process-graph' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'metabolism'  (local:MetabolismGraph)
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['summary'] = 'Metabolism — substrates + enzymes → products'
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['contract']['status'] = 'draft - no update dynamics yet'
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['contract']['summary'] = 'Metabolism — substrates + enzymes → products'
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['contract']['description'] = 'The process-graph view of metabolism: consumes substrate metabolites under enzyme (protein) catalysis to make product metabolites.'
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['contract']['math'] = ['\\frac{d[\\text{products}]}{dt} = k_{\\text{cat}}\\,[\\text{enzymes}]\\,\\frac{[\\text{substrates}]}{K_m + [\\text{substrates}]}']
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['contract']['symbols']['k_cat'] = 'turnover number'
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['contract']['symbols']['K_m'] = 'Michaelis constant'
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['contract']['ports']['substrates'] = 'substrate metabolites'
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['contract']['ports']['enzymes'] = 'enzymes (protein)'
spec_spatio_flux_composites_fig05a_process_graph['state']['metabolism']['config']['contract']['ports']['products'] = 'product metabolites'

# process 'gene_expression'  (local:GeneExpressionGraph)
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['interval'] = 1.0
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['summary'] = 'Gene expression — genes → protein'
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['contract']['status'] = 'draft - no update dynamics yet'
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['contract']['summary'] = 'Gene expression — genes → protein'
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['contract']['description'] = 'The process-graph view of gene expression: reads the gene (DNA) template and produces protein (the enzymes metabolism uses).'
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['contract']['math'] = ['\\frac{d[\\text{protein}]}{dt} = k_{\\text{expr}}\\,[\\text{genes}] - \\gamma\\,[\\text{protein}]']
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['contract']['symbols']['k_expr'] = 'expression rate'
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['contract']['symbols']['γ'] = 'protein turnover'
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['contract']['ports']['genes'] = 'gene template (DNA)'
spec_spatio_flux_composites_fig05a_process_graph['state']['gene_expression']['config']['contract']['ports']['protein'] = 'produced protein (enzymes)'

**Composite `spatio_flux.composites.fig05b-composite-process`** — `spec_spatio_flux_composites_fig05b_composite_process` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig05b_composite_process = load_spec(REPO / 'spatio_flux/composites/fig05b-composite-process.composite.json')
describe_spec(spec_spatio_flux_composites_fig05b_composite_process)

In [ ]:
# === Edit parameters for composite 'fig05b-composite-process' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-05 ===
STUDY = 'fig-05'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**5a — process graph**


In [ ]:
# 5a — process graph
show_viz(_render_one('image:visualizations/fig05a-process-graph.svg', {}, RUNS_DB, STUDY_YAML))

**5b — composite process**


In [ ]:
# 5b — composite process
show_viz(_render_one('image:visualizations/fig05b-composite-process.svg', {}, RUNS_DB, STUDY_YAML))

**Figure 5 (composite)**


In [ ]:
# Figure 5 (composite)
show_viz(_render_one('image:visualizations/figure_5.svg', {}, RUNS_DB, STUDY_YAML))

## Study: Fig 6 — orchestration patterns (`fig-06`)

**Purpose.** Three orchestration patterns drawn in bigraph-loom style — (a) multi-timestepping: temporal processes each updating at their own time interval, reading/writing a shared store at their update ticks; (b) workflow: a DAG of step processes over intermediate stores; (c) event-driven graph rewrite: divide / engulf / burst events restructuring environ + agent subgraphs.

**Claim.** image present


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-06 ===
STUDY = 'fig-06'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 6 (orchestration patterns)**


In [ ]:
# Figure 6 (orchestration patterns)
show_viz(_render_one('image:visualizations/figure_6.svg', {}, RUNS_DB, STUDY_YAML))

## Study: Fig 7 — spatio-flux example composites (`fig-07`)

**Purpose.** Three spatio-flux example composites, one panel each: 7.1 community dFBA (shared `fields` store + a dynamic-FBA process per species + Monod kinetics); 7.2 COMETS-style spatial dFBA (fields on a grid + diffusion-advection + a dFBA process per bin); 7.3 Brownian particles (a `particles` store driven by brownian_movement + enforce-boundaries). Each rendered in bigraph-loom.

**Claim.** image generated


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `7-1-community-dfba` | `spatio_flux.composites.fig07-1-community-dfba` | 0 | — |
| `7-2-comets` | `spatio_flux.composites.fig07-2-comets` | 0 | — |
| `7-3-brownian-particles` | `spatio_flux.composites.fig07-3-brownian-particles` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `spatio_flux.composites.fig07-1-community-dfba`** — `spec_spatio_flux_composites_fig07_1_community_dfba` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig07_1_community_dfba = load_spec(REPO / 'spatio_flux/composites/fig07-1-community-dfba.composite.json')
describe_spec(spec_spatio_flux_composites_fig07_1_community_dfba)

In [ ]:
# === Edit parameters for composite 'fig07-1-community-dfba' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'ecoli core dFBA'  (local:DynamicFBA)
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['interval'] = 1.0
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['config']['model_file'] = 'textbook'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['config']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['config']['substrate_update_reactions']['acetate'] = 'EX_ac_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['config']['kinetic_params']['glucose'] = [0.5, 1]
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['config']['kinetic_params']['acetate'] = [0.5, 2]
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['config']['bounds']['EX_o2_e']['lower'] = -2
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['config']['bounds']['EX_o2_e']['upper'] = None
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['config']['bounds']['ATPM']['lower'] = 1
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli core dFBA']['config']['bounds']['ATPM']['upper'] = 1

# process 'ecoli dFBA'  (local:DynamicFBA)
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli dFBA']['interval'] = 1.0
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli dFBA']['config']['model_file'] = 'iAF1260.xml'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli dFBA']['config']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli dFBA']['config']['substrate_update_reactions']['formate'] = 'EX_for_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli dFBA']['config']['kinetic_params']['glucose'] = [0.5, 1]
spec_spatio_flux_composites_fig07_1_community_dfba['state']['ecoli dFBA']['config']['kinetic_params']['formate'] = [0.5, 2]

# process 'cdiff dFBA'  (local:DynamicFBA)
spec_spatio_flux_composites_fig07_1_community_dfba['state']['cdiff dFBA']['interval'] = 1.0
spec_spatio_flux_composites_fig07_1_community_dfba['state']['cdiff dFBA']['config']['model_file'] = 'iCN900.xml'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['cdiff dFBA']['config']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['cdiff dFBA']['config']['substrate_update_reactions']['acetate'] = 'EX_ac_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['cdiff dFBA']['config']['kinetic_params']['glucose'] = [0.5, 1]
spec_spatio_flux_composites_fig07_1_community_dfba['state']['cdiff dFBA']['config']['kinetic_params']['acetate'] = [0.5, 2]

# process 'pputida dFBA'  (local:DynamicFBA)
spec_spatio_flux_composites_fig07_1_community_dfba['state']['pputida dFBA']['interval'] = 1.0
spec_spatio_flux_composites_fig07_1_community_dfba['state']['pputida dFBA']['config']['model_file'] = 'iJN746.xml'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['pputida dFBA']['config']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['pputida dFBA']['config']['substrate_update_reactions']['ammonium'] = 'EX_nh4_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['pputida dFBA']['config']['substrate_update_reactions']['glycolate'] = 'EX_glyclt_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['pputida dFBA']['config']['kinetic_params']['glucose'] = [1, 2]
spec_spatio_flux_composites_fig07_1_community_dfba['state']['pputida dFBA']['config']['kinetic_params']['ammonium'] = [2, 4]
spec_spatio_flux_composites_fig07_1_community_dfba['state']['pputida dFBA']['config']['kinetic_params']['glycolate'] = [0.5, 1]

# process 'yeast dFBA'  (local:DynamicFBA)
spec_spatio_flux_composites_fig07_1_community_dfba['state']['yeast dFBA']['interval'] = 1.0
spec_spatio_flux_composites_fig07_1_community_dfba['state']['yeast dFBA']['config']['model_file'] = 'iMM904.xml'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['yeast dFBA']['config']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['yeast dFBA']['config']['substrate_update_reactions']['ammonium'] = 'EX_nh4_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['yeast dFBA']['config']['kinetic_params']['glucose'] = [0.5, 1]
spec_spatio_flux_composites_fig07_1_community_dfba['state']['yeast dFBA']['config']['kinetic_params']['ammonium'] = [0.5, 1]

# process 'llactis dFBA'  (local:DynamicFBA)
spec_spatio_flux_composites_fig07_1_community_dfba['state']['llactis dFBA']['interval'] = 1.0
spec_spatio_flux_composites_fig07_1_community_dfba['state']['llactis dFBA']['config']['model_file'] = 'iNF517.xml'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['llactis dFBA']['config']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['llactis dFBA']['config']['substrate_update_reactions']['glutamate'] = 'EX_glu__L_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['llactis dFBA']['config']['substrate_update_reactions']['serine'] = 'EX_ser__L_e'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['llactis dFBA']['config']['kinetic_params']['glucose'] = [0.5, 1.25]
spec_spatio_flux_composites_fig07_1_community_dfba['state']['llactis dFBA']['config']['kinetic_params']['glutamate'] = [0.05, 0.1]
spec_spatio_flux_composites_fig07_1_community_dfba['state']['llactis dFBA']['config']['kinetic_params']['serine'] = [0.05, 0.1]

# process 'monod_kinetics'  (local:MonodKinetics)
spec_spatio_flux_composites_fig07_1_community_dfba['state']['monod_kinetics']['interval'] = 1.0
spec_spatio_flux_composites_fig07_1_community_dfba['state']['monod_kinetics']['config']['reactions']['assimilate_acetate']['reactant'] = 'acetate'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['monod_kinetics']['config']['reactions']['assimilate_acetate']['product'] = 'mass'
spec_spatio_flux_composites_fig07_1_community_dfba['state']['monod_kinetics']['config']['reactions']['assimilate_acetate']['km'] = 0.6
spec_spatio_flux_composites_fig07_1_community_dfba['state']['monod_kinetics']['config']['reactions']['assimilate_acetate']['vmax'] = 0.3
spec_spatio_flux_composites_fig07_1_community_dfba['state']['monod_kinetics']['config']['reactions']['assimilate_acetate']['yield'] = 0.2

**Composite `spatio_flux.composites.fig07-2-comets`** — `spec_spatio_flux_composites_fig07_2_comets` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig07_2_comets = load_spec(REPO / 'spatio_flux/composites/fig07-2-comets.composite.json')
describe_spec(spec_spatio_flux_composites_fig07_2_comets)

In [ ]:
# === Edit parameters for composite 'fig07-2-comets' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'spatial_dFBA'  (local:SpatialDFBA)
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['n_bins'] = [20, 20]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['mol_ids'] = ['glucose', 'acetate']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli core']['model_file'] = 'textbook'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli core']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli core']['substrate_update_reactions']['acetate'] = 'EX_ac_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli core']['kinetic_params']['glucose'] = [0.5, 1]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli core']['kinetic_params']['acetate'] = [0.5, 2]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli core']['bounds']['EX_o2_e']['lower'] = -2
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli core']['bounds']['EX_o2_e']['upper'] = None
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli core']['bounds']['ATPM']['lower'] = 1
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli core']['bounds']['ATPM']['upper'] = 1
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli']['model_file'] = 'iAF1260.xml'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli']['substrate_update_reactions']['formate'] = 'EX_for_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli']['kinetic_params']['glucose'] = [0.5, 1]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['ecoli']['kinetic_params']['formate'] = [0.5, 2]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['cdiff']['model_file'] = 'iCN900.xml'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['cdiff']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['cdiff']['substrate_update_reactions']['acetate'] = 'EX_ac_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['cdiff']['kinetic_params']['glucose'] = [0.5, 1]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['cdiff']['kinetic_params']['acetate'] = [0.5, 2]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['pputida']['model_file'] = 'iJN746.xml'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['pputida']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['pputida']['substrate_update_reactions']['ammonium'] = 'EX_nh4_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['pputida']['substrate_update_reactions']['glycolate'] = 'EX_glyclt_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['pputida']['kinetic_params']['glucose'] = [1, 2]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['pputida']['kinetic_params']['ammonium'] = [2, 4]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['pputida']['kinetic_params']['glycolate'] = [0.5, 1]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['yeast']['model_file'] = 'iMM904.xml'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['yeast']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['yeast']['substrate_update_reactions']['ammonium'] = 'EX_nh4_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['yeast']['kinetic_params']['glucose'] = [0.5, 1]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['yeast']['kinetic_params']['ammonium'] = [0.5, 1]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['llactis']['model_file'] = 'iNF517.xml'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['llactis']['substrate_update_reactions']['glucose'] = 'EX_glc__D_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['llactis']['substrate_update_reactions']['glutamate'] = 'EX_glu__L_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['llactis']['substrate_update_reactions']['serine'] = 'EX_ser__L_e'
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['llactis']['kinetic_params']['glucose'] = [0.5, 1.25]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['llactis']['kinetic_params']['glutamate'] = [0.05, 0.1]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['models']['llactis']['kinetic_params']['serine'] = [0.05, 0.1]
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][0] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][1] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][2] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][3] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][4] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][5] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][6] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][7] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][8] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][9] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][10] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][11] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][12] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][13] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][14] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][15] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][16] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][17] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][18] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']
spec_spatio_flux_composites_fig07_2_comets['state']['spatial_dFBA']['config']['model_grid'][19] = ['ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core', 'ecoli core']

# process 'diffusion'  (local:DiffusionAdvection)
spec_spatio_flux_composites_fig07_2_comets['state']['diffusion']['config']['n_bins'] = [20, 20]
spec_spatio_flux_composites_fig07_2_comets['state']['diffusion']['config']['bounds'] = [50.0, 50.0]
spec_spatio_flux_composites_fig07_2_comets['state']['diffusion']['config']['default_diffusion_rate'] = 0.1
spec_spatio_flux_composites_fig07_2_comets['state']['diffusion']['config']['default_diffusion_dt'] = 0.1
spec_spatio_flux_composites_fig07_2_comets['state']['diffusion']['config']['diffusion_coeffs']['glucose'] = 0.0
spec_spatio_flux_composites_fig07_2_comets['state']['diffusion']['config']['diffusion_coeffs']['acetate'] = 0.1
spec_spatio_flux_composites_fig07_2_comets['state']['diffusion']['config']['advection_coeffs']['glucose'] = [0, 0]
spec_spatio_flux_composites_fig07_2_comets['state']['diffusion']['config']['advection_coeffs']['acetate'] = [0, 0]
spec_spatio_flux_composites_fig07_2_comets['state']['diffusion']['config']['boundary_conditions'] = None

**Composite `spatio_flux.composites.fig07-3-brownian-particles`** — `spec_spatio_flux_composites_fig07_3_brownian_particles` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig07_3_brownian_particles = load_spec(REPO / 'spatio_flux/composites/fig07-3-brownian-particles.composite.json')
describe_spec(spec_spatio_flux_composites_fig07_3_brownian_particles)

In [ ]:
# === Edit parameters for composite 'fig07-3-brownian-particles' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'brownian_movement'  (local:BrownianMovement)
spec_spatio_flux_composites_fig07_3_brownian_particles['state']['brownian_movement']['interval'] = 0.1
spec_spatio_flux_composites_fig07_3_brownian_particles['state']['brownian_movement']['config']['bounds'] = [50.0, 50.0]
spec_spatio_flux_composites_fig07_3_brownian_particles['state']['brownian_movement']['config']['diffusion_rate'] = 0.5
spec_spatio_flux_composites_fig07_3_brownian_particles['state']['brownian_movement']['config']['advection_rate'] = [0, 0]
spec_spatio_flux_composites_fig07_3_brownian_particles['state']['brownian_movement']['config']['interval'] = 0.1

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-07 ===
STUDY = 'fig-07'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**7a-community-dfba.svg**


In [ ]:
# 7a-community-dfba.svg
show_viz(_render_one('image:visualizations/fig07-1-community-dfba.svg', {}, RUNS_DB, STUDY_YAML))

**7b-monod-kinetics.png**


In [ ]:
# 7b-monod-kinetics.png
show_viz(_render_one('image:visualizations/fig07b-monod-kinetics.png', {}, RUNS_DB, STUDY_YAML))

**7c-dfba.png**


In [ ]:
# 7c-dfba.png
show_viz(_render_one('image:visualizations/fig07c-dfba.png', {}, RUNS_DB, STUDY_YAML))

**7d-hybrid-community.png**


In [ ]:
# 7d-hybrid-community.png
show_viz(_render_one('image:visualizations/fig07d-hybrid-community.png', {}, RUNS_DB, STUDY_YAML))

**7e-comets.svg**


In [ ]:
# 7e-comets.svg
show_viz(_render_one('image:visualizations/fig07-2-comets.svg', {}, RUNS_DB, STUDY_YAML))

**7f-comets-snapshots.png**


In [ ]:
# 7f-comets-snapshots.png
show_viz(_render_one('image:visualizations/fig07f-comets-snapshots.png', {}, RUNS_DB, STUDY_YAML))

**7g-brownian-particles.svg**


In [ ]:
# 7g-brownian-particles.svg
show_viz(_render_one('image:visualizations/fig07-3-brownian-particles.svg', {}, RUNS_DB, STUDY_YAML))

**7h-brownian-traces.png**


In [ ]:
# 7h-brownian-traces.png
show_viz(_render_one('image:visualizations/fig07h-brownian-traces.png', {}, RUNS_DB, STUDY_YAML))

**Figure 7 (composite)**


In [ ]:
# Figure 7 (composite)
show_viz(_render_one('image:visualizations/figure_7.svg', {}, RUNS_DB, STUDY_YAML))

## Study: Fig 8 — spatio-flux reference model (`fig-08`)

**Purpose.** spatio-flux reference model (spatioflux_reference_demo): fields + diffusion + a Monod-kinetics array with Newtonian particles carrying per-particle dFBA (ecoli_1, ecoli_2) aggregated into particle mass, rendered in bigraph-loom.

**Claim.** image generated


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `reference` | `spatio_flux.composites.fig08-reference-model` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `spatio_flux.composites.fig08-reference-model`** — `spec_spatio_flux_composites_fig08_reference_model` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatio_flux_composites_fig08_reference_model = load_spec(REPO / 'spatio_flux/composites/fig08-reference-model.composite.json')
describe_spec(spec_spatio_flux_composites_fig08_reference_model)

In [ ]:
# === Edit parameters for composite 'fig08-reference-model' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'monod_kinetics[0,0]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,0]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[0,1]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,1]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[0,2]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,2]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[0,3]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[0,3]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[1,0]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,0]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[1,1]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,1]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[1,2]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,2]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[1,3]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[1,3]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[2,0]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,0]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[2,1]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,1]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[2,2]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,2]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[2,3]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[2,3]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[3,0]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,0]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[3,1]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,1]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[3,2]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,2]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'monod_kinetics[3,3]'  (local:MonodKinetics)
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['assimilate_glucose']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['assimilate_glucose']['product'] = 'mass'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['assimilate_glucose']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['assimilate_glucose']['vmax'] = 0.02
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['assimilate_glucose']['yield'] = 0.15
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['overflow_to_acetate']['reactant'] = 'glucose'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['overflow_to_acetate']['product'] = 'acetate'
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['overflow_to_acetate']['km'] = 1.0
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['overflow_to_acetate']['vmax'] = 0.2
spec_spatio_flux_composites_fig08_reference_model['state']['monod_kinetics[3,3]']['config']['reactions']['overflow_to_acetate']['yield'] = 0.85

# process 'diffusion'  (local:DiffusionAdvection)
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['n_bins'] = [4, 4]
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['bounds'] = [50.0, 50.0]
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['default_diffusion_rate'] = 0.1
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['default_diffusion_dt'] = 0.1
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['diffusion_coeffs']['glucose'] = 0.1
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['diffusion_coeffs']['acetate'] = 0.1
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['diffusion_coeffs']['dissolved biomass'] = 0.1
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['advection_coeffs']['glucose'] = [0, 0]
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['advection_coeffs']['acetate'] = [0, 0]
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['advection_coeffs']['dissolved biomass'] = [0, 0]
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['boundary_conditions']['default']['x']['type'] = 'periodic'
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['boundary_conditions']['default']['y']['type'] = 'neumann'
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['boundary_conditions']['glucose']['top']['type'] = 'dirichlet'
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['boundary_conditions']['glucose']['top']['value'] = 5.0
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['boundary_conditions']['acetate']['bottom']['type'] = 'dirichlet'
spec_spatio_flux_composites_fig08_reference_model['state']['diffusion']['config']['boundary_conditions']['acetate']['bottom']['value'] = 5.0

# process 'newtonian_particles'  (local:PymunkParticleMovement)
spec_spatio_flux_composites_fig08_reference_model['state']['newtonian_particles']['config']['gravity'] = -1.0
spec_spatio_flux_composites_fig08_reference_model['state']['newtonian_particles']['config']['elasticity'] = 0.1
spec_spatio_flux_composites_fig08_reference_model['state']['newtonian_particles']['config']['bounds'] = [50.0, 50.0]
spec_spatio_flux_composites_fig08_reference_model['state']['newtonian_particles']['config']['jitter_per_second'] = 0.01
spec_spatio_flux_composites_fig08_reference_model['state']['newtonian_particles']['config']['damping_per_second'] = 0.95
spec_spatio_flux_composites_fig08_reference_model['state']['newtonian_particles']['config']['friction'] = 0.9

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-08 ===
STUDY = 'fig-08'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**8a-reference-model.svg**


In [ ]:
# 8a-reference-model.svg
show_viz(_render_one('image:visualizations/fig08-reference-model.svg', {}, RUNS_DB, STUDY_YAML))

**8b-reference-snapshots.png**


In [ ]:
# 8b-reference-snapshots.png
show_viz(_render_one('image:visualizations/fig08b-reference-snapshots.png', {}, RUNS_DB, STUDY_YAML))

**Figure 8 (composite)**


In [ ]:
# Figure 8 (composite)
show_viz(_render_one('image:visualizations/figure_8.svg', {}, RUNS_DB, STUDY_YAML))